**Run evidence** · task `data_quality_gate` · run `478157748554319` · result **SUCCESS** · executed 2026-10-19 17:29 UTC on serverless.

Exported from the Databricks job run with `_helper/export_run_evidence.py`. Source notebook: `08_jobs/08b_data_quality_gate`.

# 08b — Data-quality gate for the gold layer

A task in the Lakeflow Job (`08a`). Runs after the gold tables are built and
decides whether the semantic layer and the Lakebase sync may be refreshed:

* every gold table has rows,
* primary keys are unique and not NULL,
* every SOURCING row points to an existing MATERIAL (MATNR + WERKS),
* every PURCHASE_ORDER vendor exists in SUPPLIER.

Results are appended to `_DQ_RESULTS` and published as the task value
`dq_passed` (`"true"` / `"false"`). The job's condition task reads it, so a failed
check **skips** the downstream refresh instead of publishing bad numbers.

In [ ]:
%run ../00_config

[00_config] Using namespace: max_kohler_demo_catalog.purchasing_demo


DemoConfig(
  catalog       = 'max_kohler_demo_catalog'
  schema        = 'purchasing_demo'
  namespace     = 'max_kohler_demo_catalog.purchasing_demo'
  warehouse_id  = 'd7a3d1cfbba73c0a'
  model         = 'databricks-claude-sonnet-4-5'
  genie_space_1 = 'SAP Purchasing — Sourcing & Supplier Risk'
  genie_space_2 = 'SAP Purchasing — Purchase Order Analytics'
  mas_endpoint  = 'sap-purchasing-mas'
  app_name      = 'sap-purchasing-app'
  job_name      = 'sap_purchasing_lakeflow_pipeline'
  lakebase      = 'sap-purchasing-ops'/'purchasing_ops' (UC catalog 'sap_purchasing_ops')
)


In [ ]:
from datetime import datetime, timezone

from pyspark.sql import Row

NS = CFG.namespace

PKS = {
    "MATERIAL": ["MATNR", "WERKS"],
    "SUPPLIER": ["LIFNR", "EKORG"],
    "SOURCING": ["MATNR", "WERKS", "LIFNR"],
    "PURCHASE_ORDER": ["EBELN", "EBELP"],
}

checks = []  # (check, table, failing_rows)

for t in GOLD_TABLES:
    n = spark.table(f"{NS}.{t}").count()
    checks.append(("has_rows", t, 0 if n > 0 else 1))

for t, keys in PKS.items():
    k = ", ".join(keys)
    nulls = " OR ".join(f"{c} IS NULL" for c in keys)
    dup = spark.sql(f"SELECT count(*) AS n FROM (SELECT {k} FROM {NS}.{t} GROUP BY {k} HAVING count(*) > 1)").first()["n"]
    nul = spark.sql(f"SELECT count(*) AS n FROM {NS}.{t} WHERE {nulls}").first()["n"]
    checks.append(("pk_unique", t, dup))
    checks.append(("pk_not_null", t, nul))

orphan_src = spark.sql(f"""SELECT count(*) AS n FROM {NS}.SOURCING s
    LEFT ANTI JOIN {NS}.MATERIAL m ON s.MATNR = m.MATNR AND s.WERKS = m.WERKS""").first()["n"]
checks.append(("fk_sourcing_material", "SOURCING", orphan_src))
orphan_po = spark.sql(f"""SELECT count(*) AS n FROM {NS}.PURCHASE_ORDER p
    LEFT ANTI JOIN (SELECT DISTINCT LIFNR FROM {NS}.SUPPLIER) s ON p.LIFNR = s.LIFNR""").first()["n"]
checks.append(("fk_po_supplier", "PURCHASE_ORDER", orphan_po))

In [ ]:
ts = datetime.now(timezone.utc)
rows = [Row(CHECKED_AT=ts, CHECK=c, TABLE_NAME=t, FAILING_ROWS=int(f), PASSED=(f == 0)) for c, t, f in checks]
spark.createDataFrame(rows).write.format("delta").mode("append").saveAsTable(f"{NS}._DQ_RESULTS")
display(spark.createDataFrame(rows))

passed = all(r.PASSED for r in rows)
failed = [f"{r.CHECK}:{r.TABLE_NAME}" for r in rows if not r.PASSED]
print(f"Data quality {'PASSED' if passed else 'FAILED'} — {len(rows)} checks, failing: {failed or 'none'}")

# Publish the verdict for the job's condition task (no-op outside a job run).
try:
    dbutils.jobs.taskValues.set(key="dq_passed", value="true" if passed else "false")
except Exception as e:
    print(f"(task value not set outside a job: {e})")

CHECKED_AT,CHECK,TABLE_NAME,FAILING_ROWS,PASSED
2026-10-19T17:30:05.554Z,has_rows,MATERIAL,0,True
2026-10-19T17:30:05.554Z,has_rows,SUPPLIER,0,True
2026-10-19T17:30:05.554Z,has_rows,SOURCING,0,True
2026-10-19T17:30:05.554Z,has_rows,SUPPLIER_RISK,0,True
2026-10-19T17:30:05.554Z,has_rows,PURCHASE_ORDER,0,True
2026-10-19T17:30:05.554Z,pk_unique,MATERIAL,0,True
2026-10-19T17:30:05.554Z,pk_not_null,MATERIAL,0,True
2026-10-19T17:30:05.554Z,pk_unique,SUPPLIER,0,True
2026-10-19T17:30:05.554Z,pk_not_null,SUPPLIER,0,True
2026-10-19T17:30:05.554Z,pk_unique,SOURCING,0,True


Data quality PASSED — 15 checks, failing: none


In [ ]:
dbutils.notebook.exit(f"08b_data_quality_gate: {'PASSED' if passed else 'FAILED'} — {len(rows)} checks, failing={failed}")

Notebook exited: 08b_data_quality_gate: PASSED — 15 checks, failing=[]
